# Tabelas de categorias

Relaciona as categorias de produto com os pedidos e as avaliações, a partir da camada processed. Gera duas tabelas:

| Tabela | Uma linha por | Uso |
|---|---|---|
| `order_items_category` | item de pedido | Receita, volume e participação por categoria |
| `category_reviews` | categoria | Avaliação média e atraso por categoria |

## Bibliotecas

In [ ]:
%run ../src/librarys.ipynb

## Carga da camada processed

Lê as tabelas geradas no notebook de normalização. As colunas de data precisam ser indicadas em `parse_dates`, porque o CSV não guarda o tipo da coluna.

In [ ]:
orders_enriched = pd.read_csv(
    "../data/processed/orders_enriched.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"],
)
order_items = pd.read_csv("../data/processed/order_items.csv")
products = pd.read_csv("../data/processed/products.csv")
category_translation = pd.read_csv("../data/processed/category_translation.csv")

## 1. Produtos com categoria

Junta produtos com a tradução da categoria. Dois ajustes:
- produtos **sem categoria** recebem `SEM CATEGORIA`, para não sumirem nos agrupamentos;
- as 2 categorias **sem tradução** recebem o próprio nome em português.

In [ ]:
products_category = products.merge(category_translation, on="product_category_name", how="left")

products_category["product_category_name"] = products_category["product_category_name"].fillna("SEM CATEGORIA")
products_category["product_category_name_english"] = products_category["product_category_name_english"].fillna(
    products_category["product_category_name"]
)

products_category[["product_id", "product_category_name", "product_category_name_english"]].head(10)

In [ ]:
products_category[["product_category_name", "product_category_name_english"]].isna().sum()

## 2. Categorias com pedidos

Junta cada item de pedido com a categoria do produto e com as informações do pedido (status, data, UF, avaliação e atraso). A tabela fica no nível de **item**, porque um mesmo pedido pode ter produtos de categorias diferentes.

O número de linhas deve continuar igual ao de `order_items`: se aumentar, algum join duplicou linhas.

In [ ]:
order_items_category = order_items.merge(
    products_category[["product_id", "product_category_name", "product_category_name_english"]],
    on="product_id",
    how="left",
)
order_items_category = order_items_category.merge(
    orders_enriched[["order_id", "order_status", "order_purchase_timestamp", "customer_state", "review_score", "is_late"]],
    on="order_id",
    how="left",
)

print("Itens antes:", len(order_items))
print("Itens depois:", len(order_items_category))

In [ ]:
order_items_category.head(10)

Categorias com mais itens vendidos.

In [ ]:
order_items_category["product_category_name"].value_counts().head(10)

## 3. Avaliação por categoria

Para cada categoria: quantidade de pedidos, avaliação média, percentual de pedidos atrasados e receita.

Antes de calcular a avaliação e o atraso, ficamos com **um registro por pedido e categoria**. Sem isso, um pedido com 5 itens iguais teria a avaliação contada 5 vezes, pesando mais na média do que um pedido com 1 item.

In [ ]:
pedido_categoria = order_items_category.drop_duplicates(["order_id", "product_category_name"])

category_reviews = pedido_categoria.groupby(["product_category_name", "product_category_name_english"]).agg(
    pedidos=("order_id", "nunique"),
    review_medio=("review_score", "mean"),
    pct_atraso=("is_late", "mean"),
).reset_index()

receita = order_items_category.groupby("product_category_name")["price"].sum().reset_index(name="receita")
category_reviews = category_reviews.merge(receita, on="product_category_name", how="left")

category_reviews["pct_atraso"] = category_reviews["pct_atraso"] * 100
category_reviews = category_reviews.sort_values("pedidos", ascending=False)

category_reviews.head(10)

Categorias com pior avaliação média, considerando apenas as com pelo menos 100 pedidos (categorias muito pequenas têm médias instáveis).

In [ ]:
category_reviews[category_reviews["pedidos"] >= 100].sort_values("review_medio").head(10)

Avaliação média das 15 categorias com mais pedidos.

In [ ]:
top15 = category_reviews.head(15).sort_values("review_medio")

top15.plot(kind="barh", x="product_category_name", y="review_medio", legend=False, title="Avaliação média - 15 categorias com mais pedidos")
plt.xlabel("Avaliação média")
plt.ylabel("")
plt.show()

## 4. Salvar

Grava as duas tabelas na camada processed.

In [ ]:
order_items_category.to_csv("../data/processed/order_items_category.csv", index=False)
category_reviews.to_csv("../data/processed/category_reviews.csv", index=False)